In [5]:
import pandas as pd
import duckdb

In [6]:
con = duckdb.connect("Sales_Database.duckdb")

In [7]:
tables = con.execute("SHOW TABLES").fetchall()
table_names = [t[0] for t in tables]

print(table_names)

['Customers', 'Date', 'Geography', 'Product', 'Sales', 'SalesTerritory']


In [8]:
for table in table_names:
    print(f"\n Table: {table}")

    cols = con.execute(f"""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_name = '{table}'
        ORDER BY ordinal_position
    """).df()

    display(cols)


 Table: Customers


,column_name,data_type
0,CustomerKey,BIGINT
1,GeographyKey,BIGINT
2,FirstName,VARCHAR
3,LastName,VARCHAR
4,BirthDate,VARCHAR
5,MaritalStatus,VARCHAR
6,Gender,VARCHAR
7,YearlyIncome,VARCHAR
8,NumberChildrenAtHome,BIGINT
9,EnglishEducation,VARCHAR



 Table: Date


,column_name,data_type
0,FullDate,DATE



 Table: Geography


...

,column_name,data_type
0,ProductKey,BIGINT
1,EnglishProductName,VARCHAR
2,StandardCost,VARCHAR
3,ListPrice,VARCHAR
4,Weight,DOUBLE



 Table: Sales


,column_name,data_type
0,ProductKey,BIGINT
1,CustomerKey,BIGINT
2,SalesTerritoryKey,BIGINT
3,SalesOrderNumber,VARCHAR
4,OrderQuantity,BIGINT
5,ProductStandardCost,VARCHAR
6,SalesAmount,VARCHAR
7,TaxAmt,VARCHAR
8,Freight,VARCHAR
9,OrderDate,DATE



 Table: SalesTerritory


,column_name,data_type
0,SalesTerritoryKey,BIGINT
1,SalesTerritoryRegion,VARCHAR
2,SalesTerritoryCountry,VARCHAR
3,SalesTerritoryGroup,VARCHAR


## 1. Customer purchase details
How can you show each customer’s purchases along with the products they bought?

In [9]:
con.execute("""
SELECT 
    Customers.CustomerKey,
    Customers.FirstName,
    Customers.LastName,
    sum(cast(REPLACE(REPLACE(SalesAmount, '$', ''), ',', '') as double)) as Total_Sales_Amount,
    STRING_AGG(DISTINCT product.EnglishProductName, ', ') as Products_Bought
FROM Customers
LEFT JOIN Sales 
    ON Customers.CustomerKey = Sales.CustomerKey
LEFT JOIN Product
    ON Sales.ProductKey = Product.ProductKey
Group By Customers.CustomerKey,
    Customers.FirstName,
    Customers.LastName
Order By Total_Sales_Amount Desc
""").df()

,CustomerKey,FirstName,LastName,Total_Sales_Amount,Products_Bought
0,12301,Nichole,Nara,13295.38,"Touring-1000 Yellow, 46, Road-250 Black, 52, S..."
1,12132,Kaitlyn,Henderson,13294.27,"Mountain-200 Black, 42, Road Tire Tube, Tourin..."
2,12308,Margaret,He,13269.27,"HL Road Tire, Sport-100 Helmet, Blue, AWC Logo..."
3,12131,Randall,Dominguez,13265.99,"Sport-100 Helmet, Blue, Road-150 Red, 44, Moun..."
4,12300,Adriana,Gonzalez,13242.70,"Road Bottle Cage, Mountain-200 Silver, 46, Roa..."
...,...,...,...,...,...
18479,29471,Dana,Ortega,NaN,None
18480,21085,Leslie,Vazquez,NaN,None
18481,24714,Whitney,Rodriguez,NaN,None
18482,27383,Jay,Hernandez,NaN,None


## 2. Product popularity across customers
How can you determine how many unique customers have purchased each product?

In [10]:
con.execute("""
Select Product.ProductKey,
    Product.EnglishProductName,
    count(distinct Sales.CustomerKey) as Customer_count
from Product Left Outer Join Sales
on Product.ProductKey = Sales.ProductKey
group by 
    Product.ProductKey,
    Product.EnglishProductName
order by Customer_count desc
""").df()

,ProductKey,EnglishProductName,Customer_count
0,477,Water Bottle - 30 oz.,3920
1,480,Patch Kit/8 Patches,2798
2,528,Mountain Tire Tube,2788
3,529,Road Tire Tube,2086
4,214,"Sport-100 Helmet, Red",2077
...,...,...,...
601,431,"ML Road Frame-W - Yellow, 42",0
602,124,Lock Nut 22,0
603,39,Thin-Jam Hex Nut 10,0
604,263,"LL Road Frame - Red, 44",0


## 3. Sales by country
How can you compute total sales amount per country using sales territory and geography information?

In [11]:
con.execute("""
Select 
    SalesTerritory.SalesTerritoryCountry,
    sum(cast(REPLACE(REPLACE(Sales.SalesAmount, '$', ''), ',', '') as double)) as Total_Sales_Amount
from SalesTerritory
Left Outer Join Sales
on SalesTerritory.SalesTerritoryKey = Sales.SalesTerritoryKey
group by 
    SalesTerritory.SalesTerritoryCountry
order by Total_Sales_Amount desc
""").df()

,SalesTerritoryCountry,Total_Sales_Amount
0,United States,9.370355e+06
1,Australia,9.051766e+06
2,United Kingdom,3.387491e+06
3,Germany,2.890708e+06
4,France,2.640526e+06
5,Canada,1.966991e+06


## 4. Product sales volume
How can you find the total quantity sold for each product?

In [12]:
con.execute("""
Select 
    Product.EnglishProductName,
    COALESCE(SUM(Sales.OrderQuantity), 0) as TotalOrderQuantity
    from Product
Left Outer Join Sales
on Product.ProductKey = Sales.ProductKey
group by 
    Product.EnglishProductName
order by TotalOrderQuantity Desc
""").df()

,EnglishProductName,TotalOrderQuantity
0,Water Bottle - 30 oz.,43050.0
1,Patch Kit/8 Patches,31112.0
2,Mountain Tire Tube,30321.0
3,Road Tire Tube,23177.0
4,"Sport-100 Helmet, Red",22039.0
...,...,...
499,Lock Washer 13,0.0
500,HL Road Front Wheel,0.0
501,Lock Washer 9,0.0
502,Seat Lug,0.0


## 5. Sales trend over time
How can you analyze total sales amount over time using order dates?

In [13]:
con.execute("""
Select sum(cast(REPLACE(REPLACE(SalesAmount, '$', ''), ',', '') as double)) as Total_Sales_Amount
,
    OrderDate
From Sales
Group BY OrderDate
""").df()

,Total_Sales_Amount,OrderDate
0,14477.34,2005-07-01
1,13931.52,2005-07-02
2,15012.18,2005-07-03
3,7156.54,2005-07-04
4,15012.18,2005-07-05
...,...,...
1088,69533.64,2008-06-26
1089,58080.78,2008-06-27
1090,78906.59,2008-06-28
1091,60959.78,2008-06-29


## 6. Inactive customers
How can you identify customers who have never made a purchase?

In [14]:
con.execute("""
Select 
    Customers.CustomerKey,
    Customers.FirstName,
    Customers.LastName
from Customers
Left Outer Join Sales
on Customers.CustomerKey = Sales.CustomerKey
where Sales.CustomerKey is null
""").df()

,CustomerKey,FirstName,LastName
0,11349,Mindy,Luo
1,14029,Christina,Morris
2,14514,Rachel,Jones
3,15381,Alexa,Howard
4,16024,Isaac,Hill
...,...,...,...
561,27032,Miguel,Garcia
562,27383,Jay,Hernandez
563,27640,Lauren,Thomas
564,28466,Marco,Chandra


## 7. Top products per sales territory

How can you identify the top-performing products (by revenue) within each sales territory?

In [15]:
con.execute("""
WITH SalesAggregate AS (
    Select 
        SalesTerritoryKey, 
        ProductKey, 
        sum(cast(REPLACE(REPLACE(SalesAmount, '$', ''), ',', '') as double)) as TotalSalesAmount
    from Sales 
    group by SalesTerritoryKey, ProductKey
),

RankedProductswithinTerritory AS (
    Select 
        SalesTerritoryKey,
        ProductKey,
        TotalSalesAmount,
        Rank() over (Partition by SalesTerritoryKey Order by TotalSalesAmount Desc) as ProductRank
    from SalesAggregate
),

Top3ProductswithinTerritory AS (
    Select
        st.SalesTerritoryRegion as Region,
        st.SalesTerritoryCountry as Country,
        st.SalesTerritoryGroup as "Group",
        p.EnglishProductName as TopProduct,
        s.ProductRank
    from RankedProductswithinTerritory as s
    Inner join SalesTerritory as st
        on st.SalesTerritoryKey = s.SalesTerritoryKey
    Inner join Product as p
        on p.ProductKey = s.ProductKey
    where s.ProductRank < 4
)

Select 
    Region, 
    Country, 
    "Group",
    STRING_AGG(TopProduct, ', ') as TopThreeProducts
from Top3ProductswithinTerritory as s3
group by 
    s3.Region, 
    s3.Country, 
    s3."Group"
""").df()

,Region,Country,Group,TopThreeProducts
0,Central,United States,North America,"Sport-100 Helmet, Blue, Mountain-200 Silver, 3..."
1,Southeast,United States,North America,"Mountain-200 Silver, 38, Mountain-200 Black, 3..."
2,United Kingdom,United Kingdom,Europe,"Mountain-200 Black, 42, Mountain-200 Black, 38..."
3,France,France,Europe,"Mountain-200 Silver, 42, Mountain-200 Silver, ..."
4,Northeast,United States,North America,"Mountain-200 Black, 42, Road-350-W Yellow, 40,..."
5,Northwest,United States,North America,"Road-150 Red, 62, Road-150 Red, 48, Mountain-2..."
6,Canada,Canada,North America,"Road-150 Red, 52, Road-150 Red, 62, Road-150 R..."
7,Southwest,United States,North America,"Road-150 Red, 52, Road-150 Red, 62, Road-150 R..."
8,Germany,Germany,Europe,"Mountain-200 Silver, 38, Mountain-200 Silver, ..."
9,Australia,Australia,Pacific,"Road-150 Red, 62, Road-150 Red, 48, Road-150 R..."


## 8. Customer lifetime value

How can you compute each customer’s total number of orders and total sales amount?

In [38]:
con.execute("""
    SELECT 
        b.FirstName, 
        b.LastName, 
        a.TotalSalesAmount, 
        a.Orders
    FROM (
        SELECT 
            CustomerKey,
            SUM(CAST(REPLACE(REPLACE(SalesAmount, '$', ''), ',', '') AS DOUBLE)) AS TotalSalesAmount,
            COUNT(*) AS Orders
        FROM Sales
        GROUP BY CustomerKey
    ) AS a
    INNER JOIN Customers AS b
        ON a.CustomerKey = b.CustomerKey
    ORDER BY a.Orders DESC
""").df()

,FirstName,LastName,TotalSalesAmount,Orders
0,Fernando,Barnes,1390.55,64
1,Ashley,Henderson,1381.37,61
2,Jennifer,Simmons,1010.56,59
3,Charles,Jackson,1305.66,57
4,Samantha,Jenkins,1290.33,56
...,...,...,...,...
17913,Tommy,Tang,2049.10,1
17914,Lacey,She,4.99,1
17915,Mariah,Watson,3578.27,1
17916,Miguel,Davis,3578.27,1


## 9. Shipping delay analysis

How can you measure the average number of days between order date and ship date?

In [50]:
con.execute("""
SELECT 
    AVG(DATE_DIFF('day', OrderDate, ShipDate)) AS AverageShippingDays
FROM sales
""").df()

,AverageShippingDays
0,7.0


## 10. Top customer per country

How can you identify the highest-spending customer in each country?

In [72]:
con.execute("""
WITH SalesClean AS (
    SELECT 
        s.CustomerKey,
        st.SalesTerritoryCountry,
        CAST(REPLACE(REPLACE(s.SalesAmount, '$', ''), ',', '') AS DOUBLE) AS SalesAmount
    FROM sales s
    JOIN SalesTerritory st
        ON s.SalesTerritoryKey = st.SalesTerritoryKey
),

CustomerCountrySales AS (
    SELECT 
        CustomerKey,
        SalesTerritoryCountry,
        SUM(SalesAmount) AS TotalSales
    FROM SalesClean
    GROUP BY CustomerKey, SalesTerritoryCountry
),


CustomerMainCountry AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY CustomerKey
            ORDER BY TotalSales DESC
        ) AS rn
    FROM CustomerCountrySales
),

FilteredCustomers AS (
    SELECT 
        CustomerKey,
        SalesTerritoryCountry,
        TotalSales
    FROM CustomerMainCountry
    WHERE rn = 1
),

RankedCustomers AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY SalesTerritoryCountry
            ORDER BY TotalSales DESC
        ) AS country_rank
    FROM FilteredCustomers
)

SELECT 
    r.SalesTerritoryCountry AS Country,
    c.FirstName,
    c.LastName,
    r.TotalSales
FROM RankedCustomers r
JOIN Customers c
    ON r.CustomerKey = c.CustomerKey
WHERE country_rank = 1
""").df()

,Country,FirstName,LastName,TotalSales
0,Germany,Ricky,Vazquez,10580.35
1,United States,Victoria,Stewart,6770.60
2,Australia,Meagan,Madan,8319.51
3,France,Nichole,Nara,13295.38
4,United Kingdom,Marie,Sanz,8460.25
5,Canada,Isabella,Bryant,6060.50
